# 第5回 演習：回帰の評価

## 前回からの接続

第4回では、欠測（missing data）を埋め、スケールをそろえ、カテゴリを開いて、現場から届いたままのデータを解析できる形に直しました。ただし、その良し悪しを判定していた物差しは R² ひとつきり。「生の数値扱い 0.829 とワンホット（one-hot） 0.844 ならこちら」とは言えても、「1日あたり何台ぶん外しているのか」には答えられないまま終わっています。

今日は、その物差しそのものを主役にします。RMSE・MAE・R² を並べて性格の違いを確かめ、第3回で使った交差検証（cross-validation）を今度は中から組み立てて、「単位付き・± 付き」で実力を言えるところまで持っていきます。


## 今日の分析目標

**モデルの実力を、ひいき目なしの数字で示したい。**

回帰モデルの実力は、RMSE・MAE・R²という複数の物差しで測り、交差検証で「平均±ばらつき」の形にまとめます。この演習では、RMSEを式から自分で計算し、交差検証も自分の手で回して、「単位付き・±付き」の数字を作ります。TODOに取り組みながら、最後の「目標に答えられたか」で振り返りましょう。


## 学習ゴール

この回を終えると、次のことができるようになります。

- RMSE を式の4ステップから自分で計算し、なぜ二乗してからルートを取るのかを説明できる
- RMSE・MAE・R² の性格の違いを、大外しへの敏感さと単位の有無から言い分け、目的に合った一本を選べる
- 交差検証で「平均 ± ばらつき」を出し、1回の分割の数字をそのまま報告してはいけない理由を言える
- モデルの実力を「およそ 887 台（± 50 台）ずれる」という単位付き・± 付きの形にして、業務の相手に伝えられる


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

!pip install -q japanize-matplotlib   # 図中の日本語が □ になるのを防ぐ
try:
    import japanize_matplotlib
except Exception:   # japanize が動かなくなったときの保険：同梱フォントを直接登録する
    from importlib.util import find_spec
    from matplotlib import font_manager as fm
    from pathlib import Path
    spec = find_spec('japanize_matplotlib')
    ttf = next(Path(spec.origin).parent.rglob('*.ttf'), None) if spec else None
    if ttf:
        fm.fontManager.addfont(str(ttf))
        plt.rcParams['font.family'] = fm.FontProperties(fname=str(ttf)).get_name()

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の化けを防ぐ
DATA_DIR = 'https://raw.githubusercontent.com/k0heiun0/applied_exercise/main/shared/data'   # データはこのリポジトリから読み込む
df = pd.read_csv(f'{DATA_DIR}/bike_day.csv')
feat = ['temp','atemp','hum','windspeed','season','yr',
        'mnth','holiday','weekday','workingday','weathersit']
X, y = df[feat].values, df['cnt'].values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
pred = LinearRegression().fit(X_tr, y_tr).predict(X_te)
print('予測と実際を用意しました')

### 分析の地図：今日はここ

データ解析は「① データの理解と目標の設定 → ② 前処理（preprocessing）とデータ解析 → ③ 結果の解釈と目標との整合」の3つのフェーズを回ります。今日は色の濃いところを扱います。


In [ ]:
# 図：分析の地図（全14回のどこにいるか）
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.axis('off')
phases = ['① データの理解と\n目標の設定', '② 前処理と\nデータ解析', '③ 結果の解釈と\n目標との整合']
colors = ['#0066cc', '#2a9d8f', '#e63946']
here = {2, 3}
for i, (p, c, x) in enumerate(zip(phases, colors, [0.17, 0.5, 0.83]), start=1):
    on = i in here
    ax.text(x, 0.62, p, ha='center', va='center', fontsize=13 if on else 11,
            color='white', bbox=dict(boxstyle='round,pad=0.6', facecolor=c,
                                     alpha=0.95 if on else 0.25))
for x0, x1 in [(0.29, 0.365), (0.62, 0.695)]:
    ax.annotate('', xy=(x1, 0.62), xytext=(x0, 0.62),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.show()


## 1. RMSEを式から自分で計算する

「引いて、二乗して、平均して、ルート」という4ステップの中身を、テストデータ全体で再現します。

### TODO①：RMSEを自分で計算する

ライブラリの `mean_squared_error` を使わずに、`y_te` と `pred` からRMSEを計算して表示してください。


### 深掘り：RMSE の骨子——なぜ「二乗して、ルート」なのか

TODO① でたどる「引いて、二乗して、平均して、ルート」を、式一本にまとめると次の形です。

$$
\mathrm{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}\left(y_i-\hat y_i\right)^2}
$$

記号を一語ずつほどきます。$y_i$ は $i$ 日目の**実際の**利用台数、$\hat y_i$ はモデルの**予測**、その差 $y_i-\hat y_i$ が**その日の誤差**です。$n$ はテストに使った日数（この分割では 147 日）。式の中身は、内側から順に「**引く**（誤差）→ **二乗** → **平均**（これが MSE）→ **ルート**」——TODO① の4ステップそのものです。

**なぜ二乗するのか**　誤差にはプラス（予測が小さすぎた日）とマイナス（大きすぎた日）が混じります。そのまま足すと**打ち消し合って**ゼロに近づき、日々大きく外していても「平均誤差ほぼゼロ」に見えてしまいます。二乗すればすべて非負になり、打ち消しは起きません。おまけに、大きい誤差ほど二乗で**強く効く**——RMSE が「**大外しを特に嫌う**」物差しになるのは、この二乗のせいです。

小さな例で確かめます。実際が 4000・2000・5000、予測が 3500・2600・5400 なら、誤差（$y_i-\hat y_i$）は $+500,\ -600,\ -400$。プラスとマイナスが混じるので、そのまま足すと $+500-600-400=-500$ と**打ち消し合って小さく**なり、割った平均も約 $-167$——外した「向き」の情報しか残らず、**大きさが消えて**しまいます。絶対値にすれば $500,\ 600,\ 400$ で平均（MAE）は $500$、二乗すれば $250000,\ 360000,\ 160000$ で平均（MSE）は約 **25.7 万**、RMSE は約 **507**。打ち消しを許さないぶん、絶対値も二乗も誤差の大きさをきちんと残します。さらに二乗では最大の誤差 $600$ が最小の $400$ の $600^2/400^2=2.25$ 倍効き、大外しを強調するのが RMSE の性質です。

**なぜルートをとるのか**　二乗した時点で、単位も「台数の二乗」という直感の効かない量に化けます（この分割の MSE は約 **69万**＝台数²）。ルートで単位を**台数に戻した**のが RMSE。この分割では約 **831**。だから「平均しておよそ **831 台**ずれる」と、そのまま人に伝わる言葉になります。1日あたり平均およそ 4500 台の利用に対して約 2 割ぶんのずれ、と見当もつきます。

**次節への橋渡し**　この RMSE 831 を、「モデルを作らず、いつも**テストデータの平均値 約4279 台**で答えたとき」の RMSE 約 **2003 台**（＝テストデータの利用台数の標準偏差そのもの）と比べると、モデルが誤差を 4 割ほどまで縮めているとわかります。この「**平均で答えるのと比べてどれだけ縮めたか**」を割合で表したのが、次節でそろって出てくる R² です。


In [ ]:
# TODO: 「引いて、二乗して、平均して、ルート」の4ステップで、RMSEを自分で計算してください
# ヒント: 誤差の配列を作ってから、一歩ずつ順番に変形します。平方根には np.sqrt が使えます
...

## 2. ライブラリで答え合わせ

TODO①の答え合わせも兼ねて、ライブラリで3つの指標を一気に計算します。自分の計算と一致するか確かめましょう。

### 深掘り：三つの物差しを並べて読む——R²・MAE・RMSE

この節のコードセルは三つの指標を一度に出します。**RMSE 831.3 / MAE 617.4 / R² 0.8277**。同じ予測を三つの角度から測った数字です。一つずつ、骨子の式とこの実データの値で読み解きます。

**MAE（平均絶対誤差）——素直な「ふつうのずれ」**

$$
\mathrm{MAE}=\frac{1}{n}\sum_{i=1}^{n}\left|\,y_i-\hat y_i\,\right|
$$

絶対値で符号を消してから平均するだけ。二乗しないので、大きく外した日も**等身大のまま**数えます。この分割では約 **617 台**——「典型的にはおよそ 617 台ずれる」。外れた日が一日あっても、その一日ぶんしか効かないので、**外れ値（outlier）に頑健（robust）**です。

**RMSE——大外しに敏感**　式は前の深掘りのとおり。二乗をはさむぶん、大きく外した日が強く効きます。だから同じ予測でも **RMSE（831）≥ MAE（617）** になります。この**差そのものが情報**です。比 $\mathrm{RMSE}/\mathrm{MAE}\approx 1.35$。もし全部の誤差が同じ大きさなら両者は一致し、大外しがあってばらつくほど比は 1 より大きくなります。つまり **RMSE と MAE の開き具合が、「一部の日で大きく外していないか」の目安**になります。

**外れ値ひとつでどれだけ変わるか**　二乗の効き方は、具体例にするとよくわかります。ほぼ 200 台の誤差が 100 個ある状況を作り、そのうち**たった1つ**を 5000 台の大外しに置き換えてみます。すると **RMSE は 200 → 538 へ跳ね上がる**のに、**MAE は 200 → 248 とほとんど動きません**。100 個のうち 1 個の外れが、RMSE をこれだけ動かす——「RMSE は大外しに敏感、MAE は頑健」を、この一例が言い切っています。だから、外れ値の混じりやすいデータで**典型的なずれ**を知りたいなら MAE、逆に**大外しを見逃したくない**なら RMSE、と選び分けます。

**なぜ必ず RMSE ≥ MAE なのか**　二乗してから平均してルート（二乗平均平方根）は、絶対値の単純平均（MAE）を**決して下回りません**。等号は全誤差がぴったり等しいときだけ。だから計算して **RMSE < MAE** になったら、それは指標の性質ではなく**計算ミスのサイン**です。この不等式は Jensen の不等式（凸関数と平均の関係）の一例で、証明そのものは凸関数の性質を扱う解析学の参考書に譲ります。

**R²（決定係数）——「平均より何割うまいか」**

$$
R^2 = 1-\frac{\sum_i (y_i-\hat y_i)^2}{\sum_i (y_i-\bar y)^2}
\;=\; 1-\frac{\mathrm{SS_{res}}}{\mathrm{SS_{tot}}}
$$

分子 $\mathrm{SS_{res}}$ は**モデルの残差（residual）二乗和**（この分割で約 **1.02 億**）、分母 $\mathrm{SS_{tot}}$ は**平均 $\bar y$ で答えたときの二乗和**（約 **5.89 億**）です。比はおよそ 0.172、それを 1 から引いて **0.828**。単位が約分で消えるので、台数の問題も別単位の問題も**同じ土俵で比べられる**のが R² の強みです。1 に近いほど良い、0 なら平均で答えるのと同じ、負なら平均以下。前の深掘りの「RMSE 831 対 平均 2003」と同じことを割合で言っているだけで、実際 $R^2 = 1-(\mathrm{RMSE}/\mathrm{RMSE_{平均}})^2$ が成り立ちます。

**いつ、どれを見るか**

- 元の単位で「**何台ずれるか**」を人に伝えたい → **RMSE か MAE**
- **大失敗を特に避けたい**（在庫切れ・医療の見逃しなど、大外し1発が致命的） → 大外しに敏感な **RMSE**
- **外れ値が多く、典型的なずれ**を知りたい → 頑健な **MAE**
- **単位の違う問題どうし**や別モデルと横並びにしたい → **R²**

一つだけで判断しないのが要点です。RMSE と MAE の**差**、R² の**水準**を合わせて見れば、「そこそこ当たるが時々大外しする」といった性格まで読めます。

**単一の数字は構造を隠す**　三つとも、何百日ぶんの誤差を**たった1つの数字**に畳んでいます。便利な反面、「**どの局面で**外しているか」は消えます。予測が大きい日ほど一律に外す、季節ごとに偏る、といった**模様**は、残差（実際 − 予測）を予測値に対して散布図に描くと見えてきます。理想は残差が 0 の線のまわりに**均等に散らばる**こと。偏りや模様があれば、指標の数字が同じでも改善の余地が残っています。指標（要約）と残差の図（診断）は、**併せて**見るのが丁寧な評価です。

**つまずき：R² が高い＝良いモデル、ではない**　これがこの回いちばんの落とし穴です。R² は、**当てはめるデータでなら、いくらでも上げられます**。試しに、訓練データにまったく**でたらめな乱数（random number）の列**を説明変数（explanatory variable）として1本足すだけで、訓練 R² は $0.79109 \to 0.79204$ と（ごくわずかですが）**上がります**。当てはめの自由度（degrees of freedom）を増やせば R² は**決して下がらない**からです。だから訓練データの R² だけを見て「良くなった」と喜ぶのは危険——これが**過学習（overfitting）**の入口です。本演習が R² を**テストデータ・交差検証で**測っているのは、この水増しを避けるためです。もう一つの落とし穴が**外挿（extrapolation）**。学習した範囲の外（たとえば観測にない猛暑日）では、R² がいくら高くても予測は当てになりません。R² は「**学習した範囲・データでどれだけ当てたか**」であって、「どんな状況でも当たる保証」ではないのです。

**発展（控えめに）：調整済み R²**　訓練 R² が変数を足すほど上がる弱点に、直接手を当てたのが調整済み R² です。

$$
R^2_{\text{adj}} = 1-(1-R^2)\,\frac{n-1}{n-p-1}
$$

$p$ は変数の数、$n$ はデータ数。$p$ を増やすとペナルティ $(n-1)/(n-p-1)$ が大きくなり、**役に立たない変数を足すと下がる**ように補正されます。この訓練データ（$n=584,\ p=11$）では、素の R² 0.7911 に対し調整済みは 0.7871 と**ごくわずかに下がる**だけ——11 変数はデータ数に対して十分少なく、水増しがほとんどないという意味です。ただし調整済み R² も**同じ訓練データ内**の話。本命はやはり、次節の「テスト（交差検証）で測る」です。


In [ ]:
rmse_lib = np.sqrt(mean_squared_error(y_te, pred))
mae_lib = mean_absolute_error(y_te, pred)
r2_lib = r2_score(y_te, pred)
print(f'RMSE: {rmse_lib:.1f}  → 平均およそ{rmse_lib:.0f}台ずれる')
print(f'MAE : {mae_lib:.1f}')
print(f'R²  : {r2_lib:.4f}')

## 3. 交差検証で「平均±ばらつき」を出す

ここまでの数字は1回の分割の結果なので、分け方の運が残っています。交差検証で運をならしましょう。

### TODO②：cross_val_score で5分割交差検証する

標準化（standardization）と線形（linear）回帰をまとめた `pipe`（この節のコードセルで定義済み）を使い、5分割の交差検証でR²を測って、「平均 ± ばらつき」の形で表示してください。分け方の運を残さないよう、混ぜてから分けます。


### 深掘り：なぜ「1回」では足りないのか——±の意味とばらつきの読み方

ここまでの RMSE 831・R² 0.83 は、すべて **1 回の Train/Test 分割**の数字でした。この「1 回」には**分け方の運**が残ります。たまたまテスト側に予測しやすい日が集まれば数字は良く、難しい日が集まれば悪く出る。報告する側に盛るつもりがなくても、**運で数字が上下**してしまうのです。

交差検証は、この運をならします。データを K 個に分け、テスト役を順に交代させて K 回測り、平均する。全データがちょうど一度ずつテストに回るので、特定の分割の当たり外れが平されます。

**± は捨てるノイズではなく、情報です**　5 分割で出る R² は 0.828 / 0.746 / 0.813 / 0.722 / 0.808 の 5 つ、その平均が 0.783、標準偏差（standard deviation）が 0.042。この **±0.042** は「分け方を変えると成績がこのくらい揺れる」幅、つまり**数字の信頼できる桁**を教えてくれます。「R² 0.783 ± 0.042」と書けば、0.78 前後は堅いが小数第 2 位は分割次第、と読めます。逆に平均だけ報告して ± を捨てると、「0.7832」のような**実態のない精度**を主張してしまいます。RMSE でも同じで、**887 ± 51 台**——「平均およそ 887 台ずれ、ぶれ幅は 50 台ほど」。単位付き・± 付き、これが目標に掲げた形です。

**発展：ばらつきそのものを読む**　平均が同じでも ± が大きいモデルは、データの一部に**弱点**を抱えている疑いがあります（ある分割だけ大きく外す）。だからモデルを比べるときは、平均だけでなく ± まで見て「**安定して勝っているか**」を確かめます。ここで測った ± は、この先モデルを比べる場面でそのまま土俵として使えます。

**発展：分割数 K はいくつにするか**　K を 3・5・10 と変えると、平均 R² は 0.787 / 0.783 / 0.778 とほとんど動きません。一方 ± は 0.012 / 0.042 / 0.072 と **K を増やすほど大きく**なります。K を大きくすると 1 回あたりのテストが小さくなり、各回のスコアが振れやすくなるためです（究極は 1 件だけ抜く一個抜き法＝LOO。正確ですが計算は重い）。平均の推定値は K によらず安定するので、実務では計算の軽い **5 分割か 10 分割**が定番。迷えば 5 で十分です。

**つまずき：shuffle を忘れると評価が壊れる**　`KFold` に `shuffle=True` を指定せず、`cross_val_score` に `cv=5` と数字だけ渡すと、回帰では**並び順のまま**分割されます。このデータは**日付順**に並んでいるので、各分割が特定の季節・年に偏り（冬だけで学習して夏を当てるような分割になり）、試しに shuffle を外すと R² の平均は **0.15 前後**まで崩れます。だから**混ぜてから分ける**のが基本。ただし時系列予測など、順番自体に意味があるデータでは別の作法（過去で学び未来で測る）を使います。交差検証の作法——前処理を各分割の訓練内で完結させる、shuffle の要否、時系列やグループ構造に合った分割の選び方——は、scikit-learn 公式ドキュメントの交差検証の節に図つきで整理されています。細部はそちらに譲ります。


In [ ]:
# 図：K分割交差検証のしくみ（5分割・テスト役が順に交代する）
K = 5
sizes = [len(te) for _, te in KFold(n_splits=K).split(X)]   # 各ブロックに入る日数
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.axis('off')
x0, bw, gap = 0.13, 0.128, 0.006          # ブロックの左端・幅・すきま
for j in range(K):
    ax.text(x0 + j * (bw + gap) + bw / 2, 0.93, f'ブロック{j+1}\n{sizes[j]}日',
            ha='center', va='center', fontsize=10, color='#555')
for k in range(K):
    top = 0.83 - k * 0.155
    for j in range(K):
        is_test = (j == k)
        ax.add_patch(plt.Rectangle((x0 + j * (bw + gap), top - 0.13), bw, 0.13,
                                   facecolor='#e76f51' if is_test else '#0066cc',
                                   alpha=0.9 if is_test else 0.55))
    ax.text(0.02, top - 0.065, f'{k+1}回目', ha='left', va='center', fontsize=11)
    ax.text(0.80, top - 0.065, '→ R²を1つ', ha='left', va='center', fontsize=10, color='#555')
ax.text(x0, 0.005, '青＝訓練（学習に使う）／オレンジ＝テスト（採点に使う）　'
                   '5つのR²を「平均 ± ばらつき」にまとめる', fontsize=11)
ax.set_xlim(0, 1); ax.set_ylim(-0.03, 1)
plt.tight_layout(); plt.show()


**この図の読み方**　横に並ぶ5つのブロックが、731 日を5等分したかたまりです（147 日と 146 日×4）。縦の5段が5回ぶんの採点で、各段でオレンジのブロックだけがテスト役、残りの青4つが訓練です。1回目はブロック1、2回目はブロック2……とオレンジが右へひとつずつずれ、5段を通すと**どのブロックもちょうど一度だけテストに回り、一度も訓練とテストを兼ねない**——ここが交差検証の肝です。段ごとに学習をやり直して R² をひとつ出すので、採点は5つ手に入ります。その平均が実力の推定値、ばらつきがその推定の不確かさで、上で見た 0.783 ± 0.042 はこの5つから出た数字です。なお図は5つのかたまりを左から順に並べていますが、`shuffle=True` を付けると先に行をよく混ぜてから切るので、各ブロックの中身は日付順のひと続きではなく、季節も年もまざった顔ぶれになります。


In [ ]:
pipe = Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())])

# TODO: 5分割の交差検証でR²を測り、「平均 ± ばらつき」の形で表示してください
# ヒント: 分割役の KFold をシャッフルありで作って、cross_val_score の cv に渡します
# ヒント: 5つのスコアの平均は .mean()、ばらつきは .std() で出せます
...

## 4. 答え合わせ＋RMSEでも交差検証

答え合わせも兼ねて、R²とRMSEの両方を交差検証で測ります。RMSEはscikit-learnの約束でマイナス付きで返るので、符号を反転して読みます。

### 深掘り：RMSE と MAE で交差検証する——符号の約束と、報告する数字

この節は、R² だけでなく RMSE と MAE でも交差検証を回して、目標「**単位付き・± 付き**」を完成させます。

**マイナス付きで返る約束**　`cross_val_score` は「大きいほど良い」でスコアの向きを統一します。ところが誤差指標は**小さいほど良い**ので、符号を反転して `neg_root_mean_squared_error` や `neg_mean_absolute_error` のように**負の値**で返します。だから頭にマイナスを付けて読み直します。指標名が `neg_` で始まっていたら、この約束のサインです。

**三つの ± 付き結果を並べる**

- R² : 0.783 ± 0.042
- RMSE: 887 ± 51 台
- MAE : 657 ± 35 台

1 回の分割では RMSE 831 / MAE 617 だったのが、運をならすと少し大きめに出ています。これは、さきほどの 1 回分割が**たまたま易しめ**だった、ということ。1 回の値より、この平均のほうが**実力の正直な見積もり**です。

**RMSE と MAE の差を、交差検証でも読む**　平均どうしでも RMSE（887）> MAE（657）で、比はおよそ 1.35。1 回分割の比（831/617 ≒ 1.35）とぴたり一致します。**大きく外す日が一定割合ある**という性格が、分割を変えても安定して見えるわけです。もしどれか一つの分割だけ差が異常に開いていたら、その分割に**極端な外れ日**が入ったサインとして拾えます。

**報告するなら**　相手が業務の人なら、「平均およそ **887 台（± 50 台）**ずれます」がいちばん親切です——単位付きで、そのままリスク判断に使えるから。R² 0.78 は、別モデルと横並びに比べるときの物差し。どちらか一方でなく、**両方を手元に持っておく**のが、ひいき目のない報告のコツです。


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_r2 = cross_val_score(pipe, X, y, cv=kf, scoring='r2')
cv_rmse = -cross_val_score(pipe, X, y, cv=kf, scoring='neg_root_mean_squared_error')
print(f'R²  : {cv_r2.mean():.3f} ± {cv_r2.std():.3f}')
print(f'RMSE: {cv_rmse.mean():.1f} ± {cv_rmse.std():.1f}  → 平均およそ{cv_rmse.mean():.0f}台ずれる')

### TODO③：指標を変えて比較する（MAE）

`scoring` を変えれば、同じ手順でMAEでも交差検証できます。MAEの5分割交差検証を行い、「平均 ± ばらつき」で表示して、上のRMSEと比べてください。

In [ ]:
# TODO: MAEで5分割交差検証し、「平均 ± ばらつき」で表示して、RMSEの結果と見比べてください
# ヒント: RMSEのときと同じ書き方で、scoring の名前をMAE用に変えるだけです
# ヒント: MAE用の名前も「neg_」で始まります。マイナス付きで返る約束にも注意
...

## 目標に答えられたか

- 今日の目標は「モデルの実力を、ひいき目なしの数字で示したい」でした
- TODO①の自分のRMSEと、ライブラリのRMSEは一致しましたか？
- TODO②の「平均±ばらつき」は、1回の分割の数字と比べてどうでしたか？ どちらを報告すべきでしょうか？
- TODO③のMAEは、RMSEより大きいですか、小さいですか？ その差は何を意味するでしょうか？
- 「RMSE 約887台 ± 50台」と「R² 0.78」、相手に伝えるならどちらが親切でしょうか？

## 今日の要点

- RMSE は「引いて、二乗して、平均して、ルート」。二乗は誤差の打ち消しを止めて大外しを強く効かせ、ルートは単位を台数に戻す
- MAE は等身大のずれ、RMSE は大外しに敏感。両者の開き（この予測では比 1.35）そのものが「一部の日で大きく外していないか」の情報になる
- R² は単位が約分で消えるので、別の問題や別モデルと横並びにできる。ただし訓練データで測れば変数を足すだけで上がるので、測る場所はテストか交差検証
- どの指標も何百日ぶんの誤差を1つの数字に畳んでいる。「どの局面で外したか」は残差の図でしか見えないので、要約と診断は併せて見る
- 1回の分割の数字には分け方の運が残る。交差検証はテスト役を順に交代させ、全データを一度ずつテストに回して運をならす
- ± は捨てるノイズではなく、数字の信頼できる桁を教える情報。平均だけを報告すると、実態のない精度を主張することになる
- 報告するなら「およそ 887 台（± 50 台）ずれます」——**単位付き・± 付き**が、ひいき目のない数字の形である


## 次回へ

RMSE・MAE・R² を並べて性格の違いを確かめ、交差検証で分け方の運をならしました。第3回で組んだ「分けて測る」足場の上に、今日は**ひいき目のない物差し**が載ったことになります。モデルの実力を「およそ 887 台（± 50 台）ずれます」と、単位付き・± 付きで人に示す準備は、これで整いました。

ただし、ここまでの数字はどれも「**どれだけ当たるか**」しか語りません。報告の場で必ず続けて聞かれるのは「で、**何が**利用台数を動かしているの？」です。次回から問いの形が変わります——外側から成績を測る段から、モデルの**中身を開いて読み**、それを人の言葉に翻訳する段へ。第6回では係数と並べ替え重要度を根拠に「何がどちらへ、どれだけ効くか」を語ります。そしてそこで、**額面どおりには読めない係数**に出くわすことになります。


## 課題（提出）

**提出するもの**: 応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

4節では `KFold(n_splits=5, shuffle=True, random_state=42)` で RMSE の交差検証をしました。分け方の運が本当にならされているか、乱数を変えて確かめます。
`random_state=0` の `KFold` に変えて、同じ `pipe`, `X`, `y` で RMSE の5分割交差検証を行い、5つの分割それぞれの RMSE と、平均 ± 標準偏差を表示してください。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

4節のセルの `random_state=42` を `0` に変えるだけです。各分割の値は、配列（`cv_rmse` にあたるもの）をそのまま `print` すれば見えます。

</details>


### 応用②（判断）

応用①で出した RMSE の**平均**は何台ですか。小数第1位まで答えてください（例: 123.4）。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

`neg_root_mean_squared_error` はマイナス付きで返るので、先頭に `-` を付けて符号を反転してから `.mean()` を取ります。`round(値, 1)` で小数第1位に丸められます。

</details>


### 応用③（解釈）

自転車シェアの**運営担当者**に、このモデルの実力を報告します。R²（4節で 0.78 前後）と RMSE（応用①の値）のどちらを、どんな言葉で伝えるか。乱数を変えると RMSE が 887 台から 902 台に動いたことを踏まえ、どの数字をどう丸めて伝えるかも含めて、単位「台」に触れながら、3行で書いてください。


（ここに3行程度で書く）


## 発展（任意）

### conformal prediction（MAPIE）で予測区間を出す

「明日はおよそ 4500 台」という点予測だけでは、自転車や人員をどれだけ用意するかは決めにくいものです。「3300〜5700 台」のように**幅**が付き、しかも「多くの日をまとめて見ると、9 割の日は区間に収まる」と言えると、意思決定にそのまま使えます。

RMSE は「平均でどれだけずれるか」を教えてくれますが、個々の日の区間までは作ってくれません。

**conformal prediction（適合予測）**は、モデルの種類や誤差の分布を仮定せずに、「指定した確率以上で区間に入る」という保証つきの区間を作る、近年、機械学習の現場で急速に広まった手法です。ただし較正に使った日と予測したい日が同じ条件で出ている（入れ替え可能な）ことは仮定します。

手順は単純です。訓練データの一部を**較正用**に取り分け、そこで残差の大きさの分位点（たとえば上位 10% の境目）を測り、それを予測の上下に足して区間にします。

ライブラリ MAPIE を使えば数行で書けます。


In [ ]:
# Colab には入っていないので、なければインストールする
import importlib.util
if importlib.util.find_spec('mapie') is None:
    %pip install -q mapie


In [ ]:
from mapie.regression import SplitConformalRegressor

# 訓練データ X_tr をさらに「学習用」と「較正用」に分ける
X_fit, X_cal, y_fit, y_cal = train_test_split(X_tr, y_tr, test_size=0.3, random_state=42)

# 学習用で pipe を学習 → 較正用で残差の分位点を測る → テストで区間を出す（信頼水準 90%）
cp = SplitConformalRegressor(estimator=pipe, confidence_level=0.9, prefit=False)
cp.fit(X_fit, y_fit).conformalize(X_cal, y_cal)
pt, iv = cp.predict_interval(X_te)          # pt: 点予測, iv: (件数, 2, 1) の下限・上限
lo, hi = iv[:, 0, 0], iv[:, 1, 0]

for i in range(5):
    print(f'実際 {y_te[i]:5.0f}  予測 {pt[i]:5.0f}  区間 [{lo[i]:5.0f}, {hi[i]:5.0f}]')
inside = (y_te >= lo) & (y_te <= hi)
print(f'区間に入った割合: {inside.mean():.3f}（{inside.sum()} / {len(y_te)} 日）')
print(f'区間の半幅: {((hi - lo) / 2).mean():.0f} 台')


In [ ]:
# 予測値の小さい順に並べて、区間（帯）と実際の値（点）を描く
order = np.argsort(pt)
xs = np.arange(len(pt))
plt.fill_between(xs, lo[order], hi[order], color='#0066cc', alpha=0.2, label='90% 予測区間')
plt.plot(xs, pt[order], color='#0066cc', lw=2, label='点予測')
plt.scatter(xs, y_te[order], s=12, color='#e63946', label='実際の利用台数')
plt.xlabel('テストデータの日（予測値の小さい順）')
plt.ylabel('利用台数')
plt.legend()
plt.show()


**読み方**　信頼水準 90% で作った区間に、テストデータ 147 日のうち 132 日（89.8%）が入りました。指定した「9 割」とほぼ一致します。保証は「多くの日をまとめて見たときの割合が 9 割以上」という平均的なもので、ある特定の日に 9 割で入るという意味ではありません。147 日での 89.8% は、その保証が成り立っている様子です。

区間の半幅はどの日も同じ 1249 台です。標準の方式（`conformity_score='absolute'`、省略時の設定）は残差の絶対値の分位点をそのまま幅にするので、日によって幅は変わりません。

4節の RMSE 約 887 台と比べると、半幅 1249 台はその 1.4 倍ほど。「平均のずれ」を伝える RMSE と、「9 割の日を囲う幅」を伝える予測区間は、同じモデルでも見せる数字が違います。

図では、赤い点のほとんどが帯の中に収まり、帯の外に出た日（約 1 割）は予測値の大小にかかわらず散らばっています。予測の小さい日では帯の下限がマイナスに食い込みますが、台数は 0 未満にならないので、そこは 0 と読み替えます。

試すなら `confidence_level` を 0.8 や 0.95 に変えて、半幅と「入った割合」がどう動くかを見てください。
